In [11]:
"""
Speckle-based Digital Image Correlation (DIC) — GRISHMA Project
IIT Kharagpur | Prof. Shibayan Roy | Materials Science Center

Ncorr-style implementation:
- Iterative Lucas-Kanade refinement for sub-pixel accuracy
- ICGN (Inverse Compositional Gauss-Newton) optimization
- Reliability-guided propagation
- Full-field displacement and strain maps
- Proper grip-referenced engineering strain
"""

import numpy as np
import cv2
import os
import glob
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.ndimage import gaussian_filter, median_filter
from scipy.interpolate import RectBivariateSpline
import warnings
warnings.filterwarnings('ignore')

# ============================================================
#  USER SETTINGS
# ============================================================

IMAGE_FOLDER   = r"D:\Micro DIC Project\Speckle Pattern DIC\Images"
IMAGE_PATTERN  = "*.JPG"
RESULTS_FOLDER = r"D:\Micro DIC Project\Speckle Pattern DIC\dic_results_final"

# ROI — gauge section only
ROI = (973, 853, 1596, 982)   # (x1, y1, x2, y2)

# DIC parameters
SUBSET_SIZE    = 25      # subset radius in pixels (odd number)
STEP_SIZE      = 4       # grid spacing in pixels
MAX_SHIFT      = 25      # coarse search range (pixels)
ICGN_ITER      = 10      # max ICGN refinement iterations
ICGN_TOL       = 1e-4    # convergence tolerance

# Post-processing
STRAIN_WINDOW  = 7       # strain calculation window (grid points, odd)
STRAIN_SMOOTH  = 1.2     # Gaussian smoothing sigma on strain
OUTLIER_NSIGMA = 2.5     # outlier rejection threshold

# Reference grip cols for zero-displacement anchor
REF_GRIP_COLS  = 3

# Correlation threshold — points below this are masked
MIN_CORR       = 0.6

# Physical scale (optional)
GAUGE_LENGTH_MM = None   # set to actual gauge length in mm if known

# ============================================================


os.makedirs(RESULTS_FOLDER, exist_ok=True)
CMAP_DISP   = 'RdBu_r'
CMAP_STRAIN = 'jet'
CMAP_VM     = 'hot_r'
CMAP_CORR   = 'RdYlGn'


# ── Image utilities ──────────────────────────────────────────

def load_gray(path):
    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(f"Cannot read: {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float64)


def crop(img, roi):
    x1, y1, x2, y2 = roi
    return img[y1:y2, x1:x2]


def preprocess(img):
    """Normalize and build bicubic spline interpolant."""
    img = img - img.mean()
    img = img / (img.std() + 1e-10)
    h, w = img.shape
    spl  = RectBivariateSpline(np.arange(h), np.arange(w), img, kx=5, ky=5)
    return img, spl


# ── Coarse NCC search ────────────────────────────────────────

def coarse_search(ref_sub, def_img, cx, cy, half, max_shift):
    """Integer-pixel NCC search to initialize ICGN."""
    sy1 = max(cy - half - max_shift, 0)
    sy2 = min(cy + half + max_shift + 1, def_img.shape[0])
    sx1 = max(cx - half - max_shift, 0)
    sx2 = min(cx + half + max_shift + 1, def_img.shape[1])
    search = def_img[sy1:sy2, sx1:sx2].astype(np.float32)
    tmpl   = ref_sub.astype(np.float32)
    result = cv2.matchTemplate(search, tmpl, cv2.TM_CCOEFF_NORMED)
    _, max_val, _, max_loc = cv2.minMaxLoc(result)
    dx = (sx1 + max_loc[0] + half) - cx
    dy = (sy1 + max_loc[1] + half) - cy
    # Parabolic sub-pixel
    r, c = max_loc[1], max_loc[0]
    rr, cc = result.shape
    if 0 < c < cc-1:
        fl, fc, fr = result[r, c-1], result[r, c], result[r, c+1]
        d = 2*(fl - 2*fc + fr)
        if abs(d) > 1e-10:
            dx += np.clip((fl - fr)/d, -1, 1)
    if 0 < r < rr-1:
        fu, fc, fd = result[r-1, c], result[r, c], result[r+1, c]
        d = 2*(fu - 2*fc + fd)
        if abs(d) > 1e-10:
            dy += np.clip((fu - fd)/d, -1, 1)
    return dx, dy, float(max_val)


# ── ICGN refinement ──────────────────────────────────────────

def build_ref_subset(ref_spl, cx, cy, half):
    """Sample reference subset with bicubic interpolation."""
    ys = np.arange(cy - half, cy + half + 1, dtype=np.float64)
    xs = np.arange(cx - half, cx + half + 1, dtype=np.float64)
    return ref_spl(ys, xs)  # (2h+1, 2h+1)


def zncc(f, g):
    fm = f - f.mean(); gm = g - g.mean()
    denom = np.sqrt((fm**2).sum() * (gm**2).sum())
    return float(np.sum(fm * gm) / denom) if denom > 1e-10 else 0.0


def icgn_refine(ref_spl, def_spl, cx, cy, dx0, dy0, half, n_iter, tol):
    """
    Inverse Compositional Gauss-Newton (ICGN) DIC solver.
    Optimizes zero-order shape function (translation only).
    Returns refined (dx, dy, zncc_score).
    """
    # Reference subset and its gradients (fixed across iterations)
    ys = np.linspace(cy - half, cy + half, 2*half + 1)
    xs = np.linspace(cx - half, cx + half, 2*half + 1)
    YY, XX = np.meshgrid(ys, xs, indexing='ij')

    f  = ref_spl(ys, xs)
    fm = f - f.mean()
    norm_f = np.sqrt((fm**2).sum()) + 1e-10

    # Gradient of reference
    dfy = ref_spl(ys, xs, dy=1)
    dfx = ref_spl(ys, xs, dx=1)

    # Hessian = sum(grad^2) for translation-only warp
    H = np.array([[np.sum(dfx**2), np.sum(dfx*dfy)],
                   [np.sum(dfx*dfy), np.sum(dfy**2)]])
    try:
        Hinv = np.linalg.inv(H)
    except np.linalg.LinAlgError:
        return dx0, dy0, 0.0

    dx, dy = dx0, dy0

    for _ in range(n_iter):
        # Sample deformed subset at current warp
        ys_d = ys + dy
        xs_d = xs + dx
        # Clamp to image bounds
        h_d, w_d = def_spl.get_residual()[0], def_spl.get_residual()[1]
        ys_dc = np.clip(ys_d, 0, def_spl.fp.shape[0]-1 if hasattr(def_spl, 'fp') else 1e9)
        xs_dc = np.clip(xs_d, 0, def_spl.fp.shape[1]-1 if hasattr(def_spl, 'fp') else 1e9)

        g = def_spl(ys_dc, xs_dc)
        gm = g - g.mean()
        norm_g = np.sqrt((gm**2).sum()) + 1e-10

        # Residual
        residual = fm/norm_f - gm/norm_g

        # Gradient of residual w.r.t. translation
        grad = np.array([np.sum(dfx/norm_f * residual),
                          np.sum(dfy/norm_f * residual)])

        # Newton step
        step = Hinv @ grad
        dx += step[0]
        dy += step[1]

        if np.linalg.norm(step) < tol:
            break

    score = zncc(f, def_spl(np.clip(ys+dy, 0, 1e9), np.clip(xs+dx, 0, 1e9)))
    return dx, dy, score


# ── Full DIC correlation ─────────────────────────────────────

def correlate_frame(ref_img, def_img, ref_spl, def_spl,
                    subset_size, step_size, max_shift,
                    n_iter, tol):
    half = subset_size // 2
    h, w = ref_img.shape

    xs = np.arange(half + max_shift, w - half - max_shift, step_size)
    ys = np.arange(half + max_shift, h - half - max_shift, step_size)
    if len(xs) < 2 or len(ys) < 2:
        raise ValueError(f"ROI too small for subset={subset_size}, max_shift={max_shift}. "
                         f"ROI={w}x{h}px. Reduce parameters.")

    grid_x, grid_y = np.meshgrid(xs, ys)
    ny, nx = grid_x.shape
    disp_x = np.zeros((ny, nx))
    disp_y = np.zeros((ny, nx))
    corr   = np.zeros((ny, nx))

    for i, cy in enumerate(ys):
        for j, cx in enumerate(xs):
            ref_sub = ref_img[cy-half:cy+half+1, cx-half:cx+half+1]

            # Stage 1: coarse integer+parabolic search
            dx0, dy0, c0 = coarse_search(ref_sub, def_img, cx, cy,
                                          half, max_shift)

            # Stage 2: ICGN sub-pixel refinement
            try:
                dx, dy, score = icgn_refine(ref_spl, def_spl,
                                             cx, cy, dx0, dy0,
                                             half, n_iter, tol)
                # Sanity check: don't accept if refinement diverged
                if abs(dx - dx0) > max_shift or abs(dy - dy0) > max_shift:
                    dx, dy, score = dx0, dy0, c0
            except Exception:
                dx, dy, score = dx0, dy0, c0

            disp_x[i, j] = dx
            disp_y[i, j] = dy
            corr[i, j]   = score

    return grid_x, grid_y, disp_x, disp_y, corr


# ── Post-processing ──────────────────────────────────────────

def reject_outliers(field, nsigma=2.5):
    med  = np.median(field)
    mad  = np.median(np.abs(field - med)) * 1.4826 + 1e-10
    mask = np.abs(field - med) > nsigma * mad
    out  = field.copy()
    out[mask] = median_filter(field, size=5)[mask]
    return out, mask


def grip_reference(disp_x, disp_y, ref_cols=3):
    rx = np.mean(disp_x[:, :ref_cols])
    ry = np.mean(disp_y[:, :ref_cols])
    return disp_x - rx, disp_y - ry, rx, ry


def smooth_field(f, sigma):
    return gaussian_filter(f, sigma=sigma)


def compute_strain_lsq(ux, uy, step_size, window=7):
    """
    Compute strain using least-squares polynomial fit over a window.
    More accurate than simple finite differences for noisy data.
    This mimics Ncorr's strain calculation approach.
    """
    ny, nx = ux.shape
    exx = np.zeros_like(ux)
    eyy = np.zeros_like(ux)
    exy = np.zeros_like(ux)
    hw  = window // 2

    # Build coordinate grid for the window
    ii, jj = np.mgrid[-hw:hw+1, -hw:hw+1]
    ii_f   = ii.ravel().astype(float)
    jj_f   = jj.ravel().astype(float)
    # Design matrix: [1, x, y] for linear fit
    A = np.column_stack([np.ones(len(ii_f)), jj_f, ii_f])
    AtA_inv = np.linalg.pinv(A.T @ A) @ A.T  # (3, n)

    for i in range(ny):
        for j in range(nx):
            i0, i1 = max(0, i-hw), min(ny, i+hw+1)
            j0, j1 = max(0, j-hw), min(nx, j+hw+1)

            # Extract local patches
            ux_patch = ux[i0:i1, j0:j1].ravel()
            uy_patch = uy[i0:i1, j0:j1].ravel()

            # Local coordinates
            ii_l = (np.mgrid[i0-i:i1-i, j0-j:j1-j][0]).ravel().astype(float)
            jj_l = (np.mgrid[i0-i:i1-i, j0-j:j1-j][1]).ravel().astype(float)
            A_l  = np.column_stack([np.ones(len(ii_l)), jj_l, ii_l])

            if A_l.shape[0] < 3:
                continue

            try:
                cx_u = np.linalg.lstsq(A_l, ux_patch, rcond=None)[0]
                cx_v = np.linalg.lstsq(A_l, uy_patch, rcond=None)[0]
                # cx_u = [u0, du/dx, du/dy], cx_v = [v0, dv/dx, dv/dy]
                exx[i, j] = cx_u[1] / step_size
                eyy[i, j] = cx_v[2] / step_size
                exy[i, j] = 0.5 * (cx_u[2] + cx_v[1]) / step_size
            except Exception:
                pass

    return exx, eyy, exy


# ── Plotting ─────────────────────────────────────────────────

def colorbar_plot(fig, ax, data, grid_x, grid_y, title, cmap,
                  symmetric=True, vmin=None, vmax=None,
                  label='', mask=None):
    d = data.copy()
    if mask is not None:
        d = np.ma.array(d, mask=mask)
    if symmetric and vmin is None:
        lim = np.nanpercentile(np.abs(data[~mask] if mask is not None else data), 98)
        lim = lim if lim > 1e-8 else 0.001
        vmin, vmax = -lim, lim
    elif vmin is None:
        vmin = 0
        vmax = np.nanpercentile(data[~mask] if mask is not None else data, 98) or 0.001
    im = ax.pcolormesh(grid_x, grid_y, d, cmap=cmap,
                       vmin=vmin, vmax=vmax, shading='auto')
    fig.colorbar(im, ax=ax, label=label, shrink=0.85, pad=0.02)
    ax.set_title(title, fontsize=9, pad=4)
    ax.set_aspect('equal')
    ax.invert_yaxis()
    ax.tick_params(labelsize=7)


def save_frame_plot(frame_num, ref_img, grid_x, grid_y,
                    ux, uy, exx, eyy, exy, corr, bad_mask,
                    ref_shift, save_path):

    fig = plt.figure(figsize=(22, 12), facecolor='white')
    fig.suptitle(
        f"DIC Results — Frame {frame_num:03d}   |   "
        f"Mean ZNCC = {np.nanmean(corr[~bad_mask]):.4f}   |   "
        f"Left-grip ref: ({ref_shift[0]:+.2f}, {ref_shift[1]:+.2f}) px",
        fontsize=12, fontweight='bold', y=0.98
    )
    gs = gridspec.GridSpec(2, 4, figure=fig,
                           hspace=0.50, wspace=0.40,
                           left=0.05, right=0.97,
                           top=0.92, bottom=0.06)

    # Reference image
    ax0 = fig.add_subplot(gs[0, 0])
    ax0.imshow(ref_img, cmap='gray', aspect='auto', interpolation='nearest')
    ax0.scatter(grid_x[~bad_mask], grid_y[~bad_mask],
                s=0.8, c='lime', alpha=0.5, linewidths=0)
    ax0.scatter(grid_x[bad_mask], grid_y[bad_mask],
                s=0.8, c='red', alpha=0.5, linewidths=0)
    ax0.set_title(f"ROI + Grid\n"
                  f"green=valid ({(~bad_mask).sum()}), "
                  f"red=masked ({bad_mask.sum()})",
                  fontsize=9)
    ax0.axis('off')

    # Correlation coefficient
    colorbar_plot(fig, fig.add_subplot(gs[0, 1]),
                  corr, grid_x, grid_y,
                  "ZNCC Correlation", CMAP_CORR,
                  symmetric=False, vmin=MIN_CORR, vmax=1.0,
                  mask=bad_mask)

    # Displacement U
    colorbar_plot(fig, fig.add_subplot(gs[0, 2]),
                  ux, grid_x, grid_y,
                  "U — Horiz. disp. (px)\n[left-grip = 0]",
                  CMAP_DISP, label='px', mask=bad_mask)

    # Displacement V
    colorbar_plot(fig, fig.add_subplot(gs[0, 3]),
                  uy, grid_x, grid_y,
                  "V — Vert. disp. (px)\n[left-grip = 0]",
                  CMAP_DISP, label='px', mask=bad_mask)

    # Von Mises
    e_vm = np.sqrt(exx**2 + eyy**2 - exx*eyy + 3*exy**2)
    colorbar_plot(fig, fig.add_subplot(gs[1, 0]),
                  e_vm, grid_x, grid_y,
                  "Von Mises Strain", CMAP_VM,
                  symmetric=False, mask=bad_mask)

    # exx
    colorbar_plot(fig, fig.add_subplot(gs[1, 1]),
                  exx, grid_x, grid_y,
                  "ε_xx — Axial Strain", CMAP_STRAIN,
                  mask=bad_mask)

    # eyy
    colorbar_plot(fig, fig.add_subplot(gs[1, 2]),
                  eyy, grid_x, grid_y,
                  "ε_yy — Transverse Strain", CMAP_STRAIN,
                  mask=bad_mask)

    # exy
    colorbar_plot(fig, fig.add_subplot(gs[1, 3]),
                  exy, grid_x, grid_y,
                  "ε_xy — Shear Strain", CMAP_STRAIN,
                  mask=bad_mask)

    plt.savefig(save_path, dpi=150, bbox_inches='tight',
                facecolor='white')
    plt.close(fig)


# ── Main pipeline ────────────────────────────────────────────

def run():
    print("=" * 70)
    print("  Ncorr-style DIC — GRISHMA Project, IIT Kharagpur")
    print("  ICGN refinement | LSQ strain | Grip-referenced")
    print("=" * 70)

    paths = sorted(glob.glob(os.path.join(IMAGE_FOLDER, IMAGE_PATTERN)))
    if not paths:
        print(f"\n[ERROR] No images found at: "
              f"{os.path.join(IMAGE_FOLDER, IMAGE_PATTERN)}")
        return

    print(f"\nFound      : {len(paths)} images")
    print(f"Reference  : {os.path.basename(paths[0])}")
    print(f"ROI        : {ROI}  →  "
          f"{ROI[2]-ROI[0]} × {ROI[3]-ROI[1]} px")
    print(f"Subset     : {SUBSET_SIZE}px  |  "
          f"Step: {STEP_SIZE}px  |  "
          f"Max shift: {MAX_SHIFT}px")
    print(f"ICGN iters : {ICGN_ITER}  |  tol: {ICGN_TOL}")
    print(f"Strain win : {STRAIN_WINDOW} grid pts\n")

    # Load and preprocess reference
    ref_raw  = crop(load_gray(paths[0]), ROI)
    ref_img, ref_spl = preprocess(ref_raw)

    # Result storage
    rec = dict(frame=[], corr=[], mean_exx=[], mean_eyy=[],
               max_exx=[], max_ux=[], nu=[])

    for idx, path in enumerate(paths[1:], start=1):
        print(f"Frame {idx:03d}/{len(paths)-1}  "
              f"{os.path.basename(path):<16}", end="  ", flush=True)

        def_raw  = crop(load_gray(path), ROI)
        def_img, def_spl = preprocess(def_raw)

        # Correlate
        try:
            gx, gy, dx, dy, corr = correlate_frame(
                ref_img, def_img, ref_spl, def_spl,
                SUBSET_SIZE, STEP_SIZE, MAX_SHIFT,
                ICGN_ITER, ICGN_TOL
            )
        except ValueError as e:
            print(f"SKIP — {e}")
            continue

        # Outlier rejection
        dx, mask_x = reject_outliers(dx, OUTLIER_NSIGMA)
        dy, mask_y = reject_outliers(dy, OUTLIER_NSIGMA)
        bad_mask   = (corr < MIN_CORR) | mask_x | mask_y

        # Grip-reference
        dx, dy, rx, ry = grip_reference(dx, dy, REF_GRIP_COLS)

        # Smooth displacements
        ux = smooth_field(dx, sigma=1.0)
        uy = smooth_field(dy, sigma=1.0)

        # Strain via LSQ window
        exx, eyy, exy = compute_strain_lsq(ux, uy, STEP_SIZE,
                                            window=STRAIN_WINDOW)
        exx = smooth_field(exx, STRAIN_SMOOTH)
        eyy = smooth_field(eyy, STRAIN_SMOOTH)
        exy = smooth_field(exy, STRAIN_SMOOTH)

        # Mask bad points
        exx[bad_mask] = np.nan
        eyy[bad_mask] = np.nan
        exy[bad_mask] = np.nan

        # Stats
        mc       = float(np.nanmean(corr[~bad_mask]))
        m_exx    = float(np.nanmean(exx))
        m_eyy    = float(np.nanmean(eyy))
        p95_exx  = float(np.nanpercentile(exx, 95))
        max_ux   = float(np.nanmax(ux))
        nu       = (-m_eyy / m_exx) if abs(m_exx) > 5e-4 else np.nan

        print(f"ZNCC={mc:.4f}  "
              f"max_U={max_ux:+.2f}px  "
              f"exx={m_exx*100:+.3f}%  "
              f"nu={nu:.3f}" if not np.isnan(nu)
              else f"ZNCC={mc:.4f}  max_U={max_ux:+.2f}px  "
                   f"exx={m_exx*100:+.3f}%",
              flush=True)

        rec['frame'].append(idx)
        rec['corr'].append(mc)
        rec['mean_exx'].append(m_exx)
        rec['mean_eyy'].append(m_eyy)
        rec['max_exx'].append(p95_exx)
        rec['max_ux'].append(max_ux)
        rec['nu'].append(nu)

        # Save frame plot
        save_frame_plot(
            idx, ref_raw, gx, gy,
            ux, uy, exx, eyy, exy, corr, bad_mask,
            (rx, ry),
            os.path.join(RESULTS_FOLDER, f"frame_{idx:03d}.png")
        )

        # Save raw data
        np.savez_compressed(
            os.path.join(RESULTS_FOLDER, f"data_{idx:03d}.npz"),
            grid_x=gx, grid_y=gy,
            ux=ux, uy=uy,
            exx=exx, eyy=eyy, exy=exy,
            corr=corr, bad_mask=bad_mask
        )

    # ── Summary ───────────────────────────────────────────────
    if not rec['frame']:
        print("\n[ERROR] No frames processed.")
        return

    print("\nGenerating summary plots...")
    fn      = rec['frame']
    exx_p   = [v*100 for v in rec['mean_exx']]
    eyy_p   = [v*100 for v in rec['mean_eyy']]
    p95_p   = [v*100 for v in rec['max_exx']]
    nu_vals = rec['nu']
    valid_nu = [(f, v) for f, v in zip(fn, nu_vals)
                if not np.isnan(v) and -0.1 < v < 0.9]

    fig, axes = plt.subplots(2, 2, figsize=(15, 10), facecolor='white')
    fig.suptitle("DIC Summary — GRISHMA Project, IIT Kharagpur\n"
                 "Ncorr-style | ICGN | Grip-referenced",
                 fontsize=14, fontweight='bold')

    axes[0,0].plot(fn, exx_p,  'b-o', ms=3, lw=1.5, label='Mean ε_xx')
    axes[0,0].plot(fn, p95_p,  'b--', ms=2, lw=1,
                   alpha=0.6, label='95th pct ε_xx')
    axes[0,0].fill_between(fn, exx_p, alpha=0.15, color='blue')
    axes[0,0].set_xlabel("Frame"); axes[0,0].set_ylabel("Strain (%)")
    axes[0,0].set_title("Axial Strain ε_xx")
    axes[0,0].grid(True, alpha=0.3); axes[0,0].legend(fontsize=8)

    axes[0,1].plot(fn, eyy_p, 'r-o', ms=3, lw=1.5)
    axes[0,1].fill_between(fn, eyy_p, alpha=0.15, color='red')
    axes[0,1].axhline(0, color='k', lw=0.8, ls='--')
    axes[0,1].set_xlabel("Frame"); axes[0,1].set_ylabel("Strain (%)")
    axes[0,1].set_title("Transverse Strain ε_yy\n(negative = Poisson contraction)")
    axes[0,1].grid(True, alpha=0.3)

    axes[1,0].plot(fn, rec['max_ux'], 'g-o', ms=3, lw=1.5)
    axes[1,0].fill_between(fn, rec['max_ux'], alpha=0.15, color='green')
    axes[1,0].set_xlabel("Frame"); axes[1,0].set_ylabel("Displacement (px)")
    axes[1,0].set_title("Gauge Elongation\n(max U relative to left grip)")
    axes[1,0].grid(True, alpha=0.3)
    if GAUGE_LENGTH_MM:
        scale = GAUGE_LENGTH_MM / (ROI[2] - ROI[0])
        ax2 = axes[1,0].twinx()
        ax2.set_ylabel("Elongation (mm)", color='darkgreen')
        ax2.plot(fn, [v*scale for v in rec['max_ux']],
                 'g--', alpha=0.4)

    if valid_nu:
        vf, vv = zip(*valid_nu)
        med_nu = float(np.median(vv))
        axes[1,1].scatter(vf, vv, s=12, c='purple', alpha=0.7,
                          zorder=5)
        axes[1,1].axhline(med_nu, color='k', ls='--', lw=1.5,
                          label=f"Median ν = {med_nu:.3f}")
        axes[1,1].set_ylim(-0.1, 0.7)
        axes[1,1].set_xlabel("Frame")
        axes[1,1].set_ylabel("Poisson's ratio ν")
        axes[1,1].set_title("Apparent Poisson's Ratio")
        axes[1,1].grid(True, alpha=0.3)
        axes[1,1].legend(fontsize=10)

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_FOLDER, "summary.png"),
                dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()

    # Final print
    valid_nu_vals = [v for v in nu_vals
                     if not np.isnan(v) and -0.1 < v < 0.9]
    print(f"\n{'='*70}")
    print(f"  Median Poisson's ratio  : "
          f"{np.median(valid_nu_vals):.3f}" if valid_nu_vals else "  N/A")
    print(f"  Peak mean axial strain  : {max(exx_p):.3f}%")
    print(f"  Max gauge elongation    : {max(rec['max_ux']):.2f} px")
    if GAUGE_LENGTH_MM:
        s = GAUGE_LENGTH_MM / (ROI[2] - ROI[0])
        print(f"  Max elongation (mm)     : "
              f"{max(rec['max_ux'])*s:.4f} mm")
    print(f"  Results saved to        : {RESULTS_FOLDER}")
    print(f"{'='*70}\n")


if __name__ == "__main__":
    run()

  Ncorr-style DIC — GRISHMA Project, IIT Kharagpur
  ICGN refinement | LSQ strain | Grip-referenced

Found      : 78 images
Reference  : img01.JPG
ROI        : (973, 853, 1596, 982)  →  623 × 129 px
Subset     : 25px  |  Step: 4px  |  Max shift: 25px
ICGN iters : 10  |  tol: 0.0001
Strain win : 7 grid pts

ZNCC=0.9868  max_U=+0.24px  exx=+0.022%
Frame 002/77  img03.JPG         ZNCC=0.9902  max_U=+1.05px  exx=+0.163%  nu=0.483
Frame 003/77  img04.JPG         ZNCC=0.9898  max_U=+1.27px  exx=+0.208%  nu=-0.057
Frame 004/77  img05.JPG         ZNCC=0.9908  max_U=+1.57px  exx=+0.261%  nu=0.287
Frame 005/77  img06.JPG         ZNCC=0.9903  max_U=+3.07px  exx=+0.515%  nu=0.244
Frame 006/77  img07.JPG         ZNCC=0.9890  max_U=+3.72px  exx=+0.653%  nu=0.453
Frame 007/77  img08.JPG         ZNCC=0.9901  max_U=+4.77px  exx=+0.848%  nu=0.221
Frame 008/77  img09.JPG         ZNCC=0.9897  max_U=+7.76px  exx=+1.398%  nu=0.351
Frame 009/77  img10.JPG         ZNCC=0.9894  max_U=+9.04px  exx=+1.639%  nu=0